# Solve the blending problem in Python

The **blending problem** — mix raw components into a product that meets a quality
specification at minimum cost — is a classic linear program: fuel blending,
feed and food formulation, alloy mixing. With **quicopt** it is a few lines in
Python.

In [1]:
%pip install "quicopt[mathopt]"

## The naive approach

Trial-and-error on the mix ratios can find *a* blend that meets spec, but not the
cheapest one, and it gets unwieldy with more components and specs. When quality
is a volume-weighted average, the problem is linear — so solve it exactly.

## Model it as an LP

A continuous variable per component is how much to use. One constraint fixes the
total volume, another enforces the minimum quality (a volume-weighted average),
and the objective minimizes cost:

In [2]:
from ortools.math_opt.python import mathopt
from quicopt import Client

# Blend components into 100 units of product at minimum cost, meeting a minimum
# quality (octane) spec.
octane = [95.0, 88.0, 92.0, 85.0]
cost   = [6.0, 4.5, 5.5, 3.8]
N = len(octane)

model = mathopt.Model(name="blending")
x = [model.add_variable(lb=0.0, name=f"comp_{i}") for i in range(N)]
model.add_linear_constraint(sum(x) == 100.0)
model.add_linear_constraint(sum(octane[i] * x[i] for i in range(N)) >= 90.0 * 100.0)
model.minimize(sum(cost[i] * x[i] for i in range(N)))

client = Client()
print(client.solve(model).display)

▄▀▀▀▄        ▄                     █
█   █ █   █ ▄▄  ▄▀▀▀▄ ▄▀▀▀▄ █▀▀▀▄ ▀█▀▀
█ ▀▄▀ █   █  █  █   ▄ █   █ █▄▄▄▀  █  ▄
 ▀▀ ▀  ▀▀▀▀ ▀▀▀  ▀▀▀   ▀▀▀  █       ▀▀

├── shots
│   ├── 1 · primal-dual · global   490.0    0.0s  ◀ best
│   └── 2 · primal-dual · global   490.0   0.01s
├── status:     optimal
├── feasible:   true
├── objective:  489.99999999999983
├── x:          comp_0=50.0, comp_1=0, comp_2=0, comp_3=50.0  (4 variables)
└── solve_time: 0.0089 s


## What you get

`status: optimal` means the blend is **proven** cheapest — cost `490` (the tiny
`…9983` tail is ordinary floating point), mixing 50 units each of the highest-
and lowest-octane components to hit exactly the 90 spec at least cost.

The same model scales to many components and several specs at once — you change
the data, not the method. (If streams mix in shared tanks, quality stops being
linear and you are in [non-convex NLP](https://quicopt.com/problems/nonconvex-nlp) territory —
which quicopt also solves.)

## Next

- The problem class behind this: [Linear programming (LP)](https://quicopt.com/problems/lp)
- A runnable model for every supported class: [Examples](https://quicopt.com/developer/tutorials/examples)
- Set up the client and solve your first model: [Getting started](https://quicopt.com/developer/getting-started)

**Reference:** H. P. Williams, *Model Building in Mathematical Programming*,
Wiley (5th ed., 2013).

## Frequently asked questions

### How is this different from the pooling problem?

Pure blending — where quality is a linear (volume-weighted) average — is an LP. True pooling, where streams mix in intermediate tanks, introduces bilinear terms and becomes a non-convex NLP. This tutorial covers the linear blending case.

### Can I add multiple quality specs or a range?

Yes — each spec (minimum octane, maximum sulfur, a target range) is another linear constraint on the same model.

### Is quicopt free to use?

Yes — pip install quicopt and your first call sets up a free key, no license.